## Sudoku Solver
My attempt at creating algorithms to solve sudokus and to get better at understanding data structures and algorithms

Sudo-Code outline
OOP Method
- Define Sudoku class
- used np arrays or maybe tensors to store the puzzle?
- Define logic rules of the game
- Create recursive solving method

Function method
- idk by now

In [ ]:
"""
GNN Sudoku Solver (pure PyTorch, no external GNN libs)

Now adapted for Jupyter Notebook usage:
- Removed argparse and __main__ block.
- Training parameters can be set as Python variables.
- Functions can be called interactively from cells.
"""
from __future__ import annotations
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import IterableDataset, DataLoader
from IPython.display import HTML, display_html

import sudoku_package as s

In [ ]:
solution = s.generate_sudoku_solution()
s.display_sudoku(solution)

puzzle = s.generate_minimal_puzzle(solution)
s.display_sudoku(puzzle)


In [ ]:
for i in range(1):
    sudoku = s.generate_sudoku_solution()

    s.tensor_to_csv(sudoku, f"training_data/4x4/{i:03d}")    

    print(sudoku)

In [ ]:
class GCNLayer(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features, bias=False)

    def forward(self, x, adj):
        """
        x: [num_nodes, in_features] (node feature matrix)
        adj: [num_nodes, num_nodes] (adjacency matrix)
        """
        # 1. Add self-loops
        I = torch.eye(adj.size(0), device=adj.device)
        A_hat = adj + I

        # 2. Compute degree matrix
        D_hat = torch.diag(torch.pow(A_hat.sum(1), -0.5))

        # 3. Normalize adjacency: D^(-1/2) * A_hat * D^(-1/2)
        A_norm = D_hat @ A_hat @ D_hat

        # 4. Apply linear transformation
        out = self.linear(x)

        # 5. Message passing: aggregate neighbor features
        out = A_norm @ out

        return out
    
class GCN(nn.Module):
    def __init__(self, in_features, hidden_features, out_features):
        super().__init__()
        self.gcn1 = GCNLayer(in_features, hidden_features)
        self.gcn2 = GCNLayer(hidden_features, out_features)

    def forward(self, x, adj):
        x = self.gcn1(x, adj)
        x = F.relu(x)
        x = self.gcn2(x, adj)
        return F.log_softmax(x, dim=1)  # for classification

In [ ]:
# Toy graph: 3 nodes, 2 edges
num_nodes = 3
adj = torch.tensor([
    [0, 1, 0],
    [1, 0, 1],
    [0, 1, 0]
], dtype=torch.float32)

# Node features: each node has 4 features
x = torch.randn(num_nodes, 4)

# Labels (e.g., node classification with 2 classes)
y = torch.tensor([0, 1, 0])

# Model
model = GCN(in_features=4, hidden_features=5, out_features=2)

# Forward pass
out = model(x, adj)
print(out)  # [num_nodes, num_classes]

In [ ]:
from torch_geometric.data import Data
from torch_geometric.utils import to_undirected

nodes = puzzle.flatten() # Not sure if I should flatten this from a 2D tensor to a 1D one or not, but for now I will?
# Might need to change this to be in the format of [[a], [b], ....]

# Every node has exactly 7 edges, idk if I actually need this or not
edges = torch.ones(16)*7

# Define sudoku squares from 0 to 15 using tuples
# NOTE: Because we are defining this using tuples, we have to call contiguous()
adj_list = torch.tensor([[0,1], [0,2], [0,3], [0,4], [0,5], [0,8], [0,12],
            [1,2], [1,3], [1,4], [1,5], [1,9], [1,13],
            [2,3], [2,6], [2,7], [2,10], [2,14],
            [3,6], [3,7], [3,11], [3,15],
            [4,5], [4,6], [4,7], [4,8], [4,12],
            [5,6], [5,7], [5,9], [5,13],
            [6,7], [6,10], [6,14],
            [7,11], [7,15],
            [8,9], [8,10], [8,11], [8,12],
            [9,10], [9,11], [9,13],
            [10,11], [10,14],
            [11,15],
            [12,13], [12,14], [12,15],
            [13,14], [13,15],
            [14,15]], dtype=torch.float32)

# NOTE: for an adjacency list {u,v} to_undirected then adds the terms {v,u}
# This is because GCNConv doesn't automatically symmetrize the adjacency list
edge_index = to_undirected(adj_list.t().contiguous())

data = Data(x=nodes, edge_index=edge_index)

# Check to make sure edge index only has values from 0 to num_nodes - 1
data.validate(raise_on_error=True)

print("Number of Nodes:", data.num_nodes)
print("Number of Edges:", data.num_edges)

In [ ]:
# Need to figure out someway to generate a sudoku dataset
# dataset = Planetoid(root='/tmp/Cora', name='Cora')

In [ ]:
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv

loader = DataLoader(data, batch_size=32, shuffle=True)


class GCN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = GCNConv(dataset.num_node_features, 16)
        self.conv2 = GCNConv(16, dataset.num_classes)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index

        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, training=self.training)
        x = self.conv2(x, edge_index)

        return F.log_softmax(x, dim=1)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GCN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

model.train()
for epoch in range(10):
    optimizer.zero_grad()
    out = model(data)
    loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()

In [ ]:
model.eval()
pred = model(data).argmax(dim=1)
correct = (pred[data.test_mask] == data.y[data.test_mask]).sum()
acc = int(correct) / int(data.test_mask.sum())
print(f'Accuracy: {acc:.4f}')